# 01 — Ingestion feux satellite (NASA FIRMS)

**Objectif** : récupérer les détections de points chauds (feux) au-dessus de la France métropolitaine via l'API *NASA FIRMS Area* (VIIRS/MODIS NRT), les enrichir, puis les écrire dans une table Delta *bronze* du Lakehouse.

> **Disclaimer — "live" = passages satellite.** Les données FIRMS NRT (Near-Real-Time) ne sont **pas** un flux continu : elles dépendent des passages des satellites (VIIRS/MODIS) au-dessus de la zone, soit quelques rafraîchissements par jour. Pour l'effet "wow" de la démo en temps réel, on utilise du **rejeu / injection** (replay) plutôt que d'attendre un passage réel.

> **Prérequis** : attacher un **Lakehouse** à ce notebook, et disposer d'une `MAP_KEY` FIRMS gratuite (https://firms.modaps.eosdis.nasa.gov/api/area/).

In [ ]:
# Cell: Paramètres
# En prod, stocke la clé dans Azure Key Vault et lis-la via notebookutils.
# MAP_KEY = notebookutils.credentials.getSecret("https://<kv>.vault.azure.net/", "firms-map-key")
MAP_KEY   = "REMPLACER_PAR_VOTRE_MAP_KEY"          # https://firms.modaps.eosdis.nasa.gov/api/area/
BBOX      = "-5.0,41.0,10.0,51.5"                   # France métropolitaine: west,south,east,north
SOURCE    = "VIIRS_NOAA20_NRT"                      # ou VIIRS_SNPP_NRT, MODIS_NRT
DAY_RANGE = 1                                        # 1 à 10 jours
LAKEHOUSE_TABLE = "bronze_fire_detections"

In [ ]:
# Cell: Appel API FIRMS + parsing CSV
import requests, io, pandas as pd
from datetime import datetime, timezone
url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/{SOURCE}/{BBOX}/{DAY_RANGE}"
resp = requests.get(url, timeout=60)
resp.raise_for_status()
df = pd.read_csv(io.StringIO(resp.text))
print(f"{len(df)} détections récupérées pour la France ({SOURCE}).")
df.head()
# Colonnes clés: latitude, longitude, frp, confidence, acq_date, acq_time, daynight, satellite

In [ ]:
# Cell: Enrichissement
df["ingest_ts"] = datetime.now(timezone.utc).isoformat()
df["source"]    = SOURCE
df["detection_id"] = (df["latitude"].round(4).astype(str) + "_" +
                      df["longitude"].round(4).astype(str) + "_" +
                      df["acq_date"].astype(str) + df["acq_time"].astype(str))
df["frp"] = pd.to_numeric(df["frp"], errors="coerce").fillna(0.0)

In [ ]:
# Cell: Écriture dans le Lakehouse (table Delta bronze)
sdf = spark.createDataFrame(df)
(sdf.write.format("delta").mode("append").saveAsTable(LAKEHOUSE_TABLE))
print(f"Écrit dans {LAKEHOUSE_TABLE}. Attache un Lakehouse à ce notebook au préalable.")

> **Temps réel (Eventstream)** : Pour du push temps réel vers l'Eventstream, remplace l'écriture Delta par un POST vers l'endpoint custom de l'Eventstream (voir README).